[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somanshusingla/llm-from-scratch/blob/main/notebooks/08_nifty50_news_llm.ipynb)

# Project — A From-Scratch LLM on Nifty 50 Stock News

**Build a Large Language Model (From Scratch)** · bonus project

This capstone applies *everything* from the course to a real, live task:

1. **Collect** recent news about the Nifty 50 companies from the internet
   (Google News RSS, which aggregates Moneycontrol, LiveMint, Economic Times, etc.,
   plus Moneycontrol/LiveMint market feeds directly).
2. **Train a GPT from scratch** (the same architecture you built) on that news
   corpus, so it learns the *language and topics* of Indian stock-market news.
3. **Answer questions about the stocks** — grounded in the collected news.

### An honest note on how the Q&A works
A small model trained on a few thousand headlines can learn the **style** of
financial news, but it cannot reliably **memorize facts**. That's not a flaw in
your code — it's a fundamental limit of small models + small data. So for the
**question-answering** part we do what production systems do: **Retrieval-Augmented
Generation (RAG)** — we retrieve the most relevant real news for your question and
answer *from that evidence*. You get both:

- a **from-scratch GPT** that generates fluent stock-news-style text, and
- a **working Q&A** that gives grounded, sourced answers about any Nifty 50 stock.

> Runs on CPU (small model, fewer stocks) and scales up automatically on a GPU
> (bigger model, all 50 stocks, more training).

## 0. Setup

In [1]:
import importlib, subprocess, sys
def ensure(pkg, name=None):
    try: importlib.import_module(name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
ensure("tiktoken")

import torch, torch.nn as nn, tiktoken, math, time, re, html, os
import urllib.request, urllib.parse
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
print("torch:", torch.__version__, "| device:", device)

torch: 2.11.0+cu128 | device: cuda


## 1. Collect Nifty 50 news from the internet

We pull recent news two ways:
- **Per-stock** via Google News RSS search (`news.google.com/rss/search?q=...`),
  which surfaces Moneycontrol, LiveMint, ET, Business Standard, etc.
- **General market feeds** directly from Moneycontrol and LiveMint RSS.

Everything uses only the Python standard library (no scraping frameworks), and
each request is wrapped in try/except so a blocked/slow source never breaks the
run. If the network is unavailable, a small built-in sample corpus is used so the
notebook still runs end-to-end.

In [2]:
# The Nifty 50 constituents (company names used for news search).
NIFTY50 = [
    "Reliance Industries", "HDFC Bank", "ICICI Bank", "Infosys", "TCS",
    "Larsen & Toubro", "ITC", "Bharti Airtel", "State Bank of India", "Axis Bank",
    "Kotak Mahindra Bank", "Hindustan Unilever", "Bajaj Finance", "Asian Paints",
    "Maruti Suzuki", "Sun Pharma", "Titan Company", "NTPC", "Power Grid", "Wipro",
    "UltraTech Cement", "Nestle India", "HCL Technologies", "Tata Motors",
    "Tata Steel", "Adani Enterprises", "Adani Ports", "JSW Steel", "Coal India",
    "Bajaj Finserv", "Tech Mahindra", "Grasim Industries", "Hindalco", "Cipla",
    "Dr Reddy's Laboratories", "Eicher Motors", "Britannia Industries", "Divi's Laboratories",
    "Apollo Hospitals", "Bajaj Auto", "Hero MotoCorp", "BPCL", "Tata Consumer Products",
    "SBI Life Insurance", "HDFC Life Insurance", "Mahindra & Mahindra", "IndusInd Bank",
    "ONGC", "Shriram Finance", "LTIMindtree",
]

def _clean(text):
    text = html.unescape(text or "")
    text = re.sub(r"<[^>]+>", " ", text)          # strip HTML tags
    text = re.sub(r"\s+", " ", text).strip()
    return text

def fetch_rss(url, max_items=40):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=20) as r:
        root = ET.fromstring(r.read())
    out = []
    for it in root.iter("item"):
        title = _clean(it.findtext("title"))
        desc = _clean(it.findtext("description"))
        if title:
            out.append({"title": title, "desc": desc, "date": _clean(it.findtext("pubDate"))})
        if len(out) >= max_items:
            break
    return out

def google_news(query, max_items=30):
    q = urllib.parse.quote(f"{query} stock share price")
    url = f"https://news.google.com/rss/search?q={q}+when:60d&hl=en-IN&gl=IN&ceid=IN:en"
    return fetch_rss(url, max_items)

In [3]:
# How much to collect: all 50 stocks on GPU, a smaller set on CPU (keeps it quick).
MAX_STOCKS = len(NIFTY50) if device.type == "cuda" else 12
PER_STOCK  = 30 if device.type == "cuda" else 12

news = []   # each: {stock, title, desc, date}
print(f"Collecting news for {MAX_STOCKS} stocks (this can take a minute)...")
for i, name in enumerate(NIFTY50[:MAX_STOCKS]):
    try:
        for it in google_news(name, PER_STOCK):
            it["stock"] = name
            news.append(it)
    except Exception as e:
        print(f"  [skip] {name}: {e}")
    if (i + 1) % 10 == 0:
        print(f"  ...{i+1} stocks, {len(news)} items so far")

# General market feeds from Moneycontrol & LiveMint (fuller descriptions).
GENERAL_FEEDS = [
    "https://www.moneycontrol.com/rss/marketreports.xml",
    "https://www.moneycontrol.com/rss/business.xml",
    "https://www.moneycontrol.com/rss/latestnews.xml",
    "https://www.livemint.com/rss/markets",
    "https://www.livemint.com/rss/companies",
]
for url in GENERAL_FEEDS:
    try:
        for it in fetch_rss(url, 40):
            it["stock"] = "Market"
            news.append(it)
    except Exception as e:
        print(f"  [skip feed] {url}: {e}")

print("Raw items collected:", len(news))

  ...10 stocks, 300 items so far


  ...20 stocks, 600 items so far


  ...30 stocks, 900 items so far


  ...40 stocks, 1200 items so far


  ...50 stocks, 1500 items so far


Raw items collected: 1604


In [4]:
# Deduplicate by title and build the final records + training corpus.
seen, items = set(), []
for it in news:
    key = it["title"].lower()
    if key in seen or len(it["title"]) < 15:
        continue
    seen.add(key)
    items.append(it)

# Fallback so the notebook always runs, even fully offline.
if len(items) < 20:
    print("Few/no items fetched (offline?) -- using a small built-in sample.")
    sample = [
        ("Reliance Industries", "Reliance Industries shares rise as Jio and retail post strong quarterly growth"),
        ("HDFC Bank", "HDFC Bank reports steady loan growth; net interest margin stable in latest quarter"),
        ("Infosys", "Infosys raises FY revenue guidance on large deal wins in financial services"),
        ("TCS", "TCS Q results beat estimates; management upbeat on demand recovery"),
        ("Tata Motors", "Tata Motors JLR volumes climb; EV sales momentum continues in India"),
        ("ICICI Bank", "ICICI Bank profit grows on higher core operating income and lower provisions"),
        ("Adani Enterprises", "Adani Enterprises gains on new airport and green hydrogen project updates"),
        ("Bharti Airtel", "Bharti Airtel ARPU improves as tariff hikes flow through subscriber base"),
        ("Maruti Suzuki", "Maruti Suzuki sales rise on strong SUV demand ahead of festive season"),
        ("SBI", "State Bank of India posts record quarterly profit on robust retail lending"),
    ]
    items = [{"stock": s, "title": t, "desc": "", "date": ""} for s, t in sample] * 12

# One text record per item: "Company | Date -- Title. Description"
def record_text(it):
    parts = [it["stock"]]
    if it["date"]:
        parts.append(it["date"])
    head = " | ".join(parts)
    body = it["title"] + (". " + it["desc"] if it["desc"] else "")
    return f"{head} -- {body}"

records = [record_text(it) for it in items]
corpus = ("\n<|endoftext|>\n").join(records)

print("Unique news items:", len(items))
print("Corpus characters:", len(corpus))
print("\nSample records:")
for r in records[:5]:
    print(" -", r[:140])

Unique news items: 1453
Corpus characters: 387133

Sample records:
 - Reliance Industries | Mon, 08 Jun 2026 07:00:00 GMT -- Stock market crash: Reliance Industries, TCS, RVNL, Wipro among 82 stocks that hit 52
 - Reliance Industries | Wed, 24 Jun 2026 15:42:53 GMT -- Reliance Industries share price to be in focus on Thursday. Here's why - Mint. Relian
 - Reliance Industries | Mon, 08 Jun 2026 07:00:00 GMT -- Reliance Industries hits 52-week low, stock down 12% in one month - Business Standard
 - Reliance Industries | Fri, 19 Jun 2026 07:00:00 GMT -- IPO: Reliance Jio announces what may be India's biggest-ever share sale - BBC. IPO: R
 - Reliance Industries | Mon, 22 Jun 2026 07:00:00 GMT -- Reliance Industries stock jumps 2% after AGM; brokerages see up to 34 percent upside 


## 2. Train a GPT from scratch on the news

We reuse the exact GPT architecture from the course (included below so this
notebook is self-contained) and train it on the news corpus. It learns the
vocabulary, phrasing, and patterns of Indian market news.

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out, self.num_heads, self.head_dim = d_out, num_heads, d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
    def forward(self, x):
        b, n, _ = x.shape
        k = self.W_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        s = q @ k.transpose(2, 3)
        s.masked_fill_(self.mask.bool()[:n, :n], -torch.inf)
        w = self.dropout(torch.softmax(s / k.shape[-1]**0.5, dim=-1))
        return self.out_proj((w @ v).transpose(1, 2).contiguous().view(b, n, self.d_out))

class LayerNorm(nn.Module):
    def __init__(self, d):
        super().__init__(); self.eps=1e-5
        self.scale=nn.Parameter(torch.ones(d)); self.shift=nn.Parameter(torch.zeros(d))
    def forward(self, x):
        m=x.mean(-1,keepdim=True); v=x.var(-1,keepdim=True,unbiased=False)
        return self.scale*(x-m)/torch.sqrt(v+self.eps)+self.shift

class GELU(nn.Module):
    def forward(self, x):
        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.044715*torch.pow(x,3))))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers=nn.Sequential(nn.Linear(cfg["emb_dim"],4*cfg["emb_dim"]),GELU(),
                                  nn.Linear(4*cfg["emb_dim"],cfg["emb_dim"]))
    def forward(self, x): return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att=MultiHeadAttention(cfg["emb_dim"],cfg["emb_dim"],cfg["context_length"],
                                    cfg["drop_rate"],cfg["n_heads"],cfg["qkv_bias"])
        self.ff=FeedForward(cfg); self.norm1=LayerNorm(cfg["emb_dim"]); self.norm2=LayerNorm(cfg["emb_dim"])
        self.drop=nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        x=x+self.drop(self.att(self.norm1(x)))
        x=x+self.drop(self.ff(self.norm2(x)))
        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"],cfg["emb_dim"])
        self.pos_emb=nn.Embedding(cfg["context_length"],cfg["emb_dim"])
        self.drop_emb=nn.Dropout(cfg["drop_rate"])
        self.trf_blocks=nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm=LayerNorm(cfg["emb_dim"])
        self.out_head=nn.Linear(cfg["emb_dim"],cfg["vocab_size"],bias=False)
    def forward(self, idx):
        b,n=idx.shape
        x=self.tok_emb(idx)+self.pos_emb(torch.arange(n,device=idx.device))
        return self.out_head(self.final_norm(self.trf_blocks(self.drop_emb(x))))

def generate(model, idx, max_new_tokens, context_size, temperature=0.8, top_k=25, eos_id=None):
    model.eval()
    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits = model(idx[:, -context_size:])[:, -1, :]
        if top_k:
            tv,_ = torch.topk(logits, top_k)
            logits = torch.where(logits < tv[:, -1], torch.tensor(float("-inf")).to(logits.device), logits)
        if temperature and temperature > 0:
            idx_next = torch.multinomial(torch.softmax(logits/temperature, dim=-1), 1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)
        if eos_id is not None and (idx_next == eos_id).all(): break
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def ids(text): return torch.tensor(tokenizer.encode(text, allowed_special={"<|endoftext|>"})).unsqueeze(0)
def txt(t): return tokenizer.decode(t.squeeze(0).tolist())
print("Model code ready.")

Model code ready.


In [6]:
# Device-aware config: a real model on GPU, a small one on CPU.
if device.type == "cuda":
    CFG = {"vocab_size":50257,"context_length":256,"emb_dim":512,"n_heads":8,"n_layers":8,"drop_rate":0.1,"qkv_bias":False}
    MAX_STEPS, BATCH = 4000, 32
else:
    CFG = {"vocab_size":50257,"context_length":128,"emb_dim":192,"n_heads":6,"n_layers":4,"drop_rate":0.1,"qkv_bias":False}
    MAX_STEPS, BATCH = 300, 16

# Tokenize the whole corpus and make a sliding-window dataset.
class TokenWindows(Dataset):
    def __init__(self, text, ctx, stride):
        t = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        self.x, self.y = [], []
        for i in range(0, max(1, len(t) - ctx), stride):
            self.x.append(torch.tensor(t[i:i+ctx])); self.y.append(torch.tensor(t[i+1:i+ctx+1]))
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i], self.y[i]

ctx = CFG["context_length"]
ds = TokenWindows(corpus, ctx, stride=ctx // 2)
loader = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
print("Total tokens:", len(tokenizer.encode(corpus, allowed_special={'<|endoftext|>'})))
print("Training windows:", len(ds), "| batches/epoch:", len(loader))

Total tokens: 106811
Training windows: 833 | batches/epoch: 26


In [7]:
torch.manual_seed(123)
model = GPTModel(CFG).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

t0, step = time.time(), 0
model.train()
while step < MAX_STEPS:
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(model(x).flatten(0, 1), y.flatten())
        loss.backward(); optimizer.step()
        step += 1
        if step % max(1, MAX_STEPS // 12) == 0 or step == 1:
            print(f"step {step:4d}/{MAX_STEPS} | loss {loss.item():.3f} | {time.time()-t0:.0f}s")
        if step >= MAX_STEPS: break
print(f"Done training in {time.time()-t0:.0f}s on {device}")

Model parameters: 76,802,048


step    1/4000 | loss 10.946 | 0s


step  333/4000 | loss 1.307 | 70s


step  666/4000 | loss 0.166 | 140s


step  999/4000 | loss 0.045 | 210s


...

step 2997/4000 | loss 0.011 | 629s


step 3330/4000 | loss 0.013 | 699s


step 3663/4000 | loss 0.013 | 769s


step 3996/4000 | loss 0.011 | 838s


Done training in 839s on cuda


In [8]:
# See what the from-scratch model generates for a few stock prompts.
for prompt in ["Reliance Industries", "HDFC Bank", "Infosys", "Nifty 50"]:
    out = generate(model, ids(prompt).to(device), max_new_tokens=30,
                   context_size=CFG["context_length"], temperature=0.7, top_k=25,
                   eos_id=tokenizer.n_vocab)   # eos disabled; just sample
    print(f"[{prompt}] ->", txt(out).replace("\n", " ")[:180])
    print("-"*80)

[Reliance Industries] -> Reliance Industries | Mon, 08 Jun 2026 07:00:00 GMT -- Stock market crash: Reliance Industries, TCS, RVNL, Wip
--------------------------------------------------------------------------------


[HDFC Bank] -> HDFC Bank | Mon, 15 Jun 2026 07:00:00 GMT -- Nifty Bank rallies 1,000 points; HDFC Bank, IndusInd
--------------------------------------------------------------------------------


[Infosys] -> Infosys: IT stocks continue to fall amid AI jitters - Mint. TCS, Wipro to Infosys: IT stocks continue to fall amid
--------------------------------------------------------------------------------


[Nifty 50] -> Nifty 50hudas L&T Q4 Results 2026: Share Price Target & Dividend Details Liquide Blog <|endoftext|> Larsen &
--------------------------------------------------------------------------------


The generated text reads like stock-market news — the model has learned the
domain's vocabulary and phrasing. (With more data/steps on a GPU it gets sharper.)
But to *answer questions* reliably, we ground responses in the real news next.

## 3. Answer questions about the stocks (Retrieval-Augmented)

We build a small **TF-IDF search index** (from scratch, no libraries) over the
news items. For any question, we retrieve the most relevant real news and answer
from that evidence — with sources. This is exactly the RAG pattern used by
production assistants.

In [9]:
import math
from collections import Counter

def tokenize_words(s):
    return re.findall(r"[a-z0-9]+", s.lower())

# Build the TF-IDF index over the news items.
docs = [it["title"] + " " + it["desc"] + " " + it["stock"] for it in items]
doc_tokens = [tokenize_words(d) for d in docs]
df = Counter()
for toks in doc_tokens:
    for w in set(toks):
        df[w] += 1
N = len(docs)
idf = {w: math.log((N + 1) / (c + 1)) + 1 for w, c in df.items()}

def tfidf_vec(tokens):
    tf = Counter(tokens)
    vec = {w: (tf[w] / len(tokens)) * idf.get(w, math.log(N + 1) + 1) for w in tf}
    norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
    return {w: v / norm for w, v in vec.items()}

doc_vecs = [tfidf_vec(t) for t in doc_tokens]

def cosine(a, b):
    return sum(a.get(w, 0.0) * b.get(w, 0.0) for w in a)

def search(query, k=5):
    qv = tfidf_vec(tokenize_words(query))
    scored = sorted(((cosine(qv, dv), i) for i, dv in enumerate(doc_vecs)), reverse=True)
    return [items[i] for s, i in scored[:k] if s > 0]

print("Search index built over", N, "news items.")

Search index built over 1453 news items.


In [10]:
def answer_question(question, k=5):
    hits = search(question, k=k)
    print("Q:", question)
    if not hits:
        print("A: I couldn't find relevant news for that question.\n"); return
    print(f"A: Based on {len(hits)} recent news item(s):")
    for h in hits:
        line = h["title"] if not h["desc"] else f'{h["title"]} — {h["desc"][:120]}'
        print(f"   • [{h['stock']}] {line}")
    print()

for q in [
    "How is Reliance Industries doing?",
    "Any news about HDFC Bank profit?",
    "What is happening with Tata Motors and EV?",
    "Latest on Infosys revenue guidance",
    "Adani group updates",
]:
    answer_question(q)

Q: How is Reliance Industries doing?
A: Based on 5 recent news item(s):
   • [Reliance Industries] Reliance Jio IPO: What will be its impact on Reliance Industries? Analysts decode - Business Today — Reliance Jio IPO: What will be its impact on Reliance Industries? Analysts decode Business Today
   • [Reliance Industries] Reliance Industries share price to be in focus on Thursday. Here's why - Mint — Reliance Industries share price to be in focus on Thursday. Here's why Mint
   • [Reliance Industries] Reliance Industries among 6 largecap stocks showing bullish RSI upswing - The Economic Times — Reliance Industries among 6 largecap stocks showing bullish RSI upswing The Economic Times
   • [Reliance Industries] Reliance Industries hits 52-week low, stock down 12% in one month - Business Standard — Reliance Industries hits 52-week low, stock down 12% in one month Business Standard
   • [Reliance Industries] Reliance Industries (NSEI:RELIANCE) Stock Fair Value Edges Higher After Analyst F

### Bonus: RAG generation (retrieval + your from-scratch model)

We can feed the retrieved news as **context** to the model and let it generate a
short continuation — combining your from-scratch GPT with grounded evidence.

In [11]:
def rag_generate(question, k=3, max_new_tokens=40):
    hits = search(question, k=k)
    context = " ".join(h["title"] for h in hits)[:600]
    prompt = f"{context}\nQuestion: {question}\nAnswer:"
    out = generate(model, ids(prompt).to(device), max_new_tokens=max_new_tokens,
                   context_size=CFG["context_length"], temperature=0.7, top_k=25,
                   eos_id=tokenizer.n_vocab)
    generated = txt(out)[len(prompt):].replace("\n", " ").strip()
    print("Q:", question)
    print("Retrieved context:", context[:160], "...")
    print("Model answer:", generated[:200])
    print("-"*80)

rag_generate("What is the outlook for Reliance Industries?")
rag_generate("How is the IT sector doing?")

Q: What is the outlook for Reliance Industries?
Retrieved context: Reliance Industries shares set for re-rating? Target price, what history suggests - Business Today Reliance Jio IPO: What will be its impact on Reliance Industr ...
Model answer: 30 -- Reliance Industries | Tue, 23 Apr 2024 AI data centre: Reliance Industries share price rises over 2% after Meta partnership with Meta Platforms tenure. What lies ahead of 49th annual
--------------------------------------------------------------------------------


Q: How is the IT sector doing?
Retrieved context: Maruti Suzuki shares jump over 4%. How is the new E100 regulation triggering a surge? - The Economic Times Jefferies blames HDFC Bank controversy for impacting  ...
Model answer: 26 quarter The Economic Times. Stocks to buy:11:02 GMT -- Maruti Suzuki India - 10s energy stocks cross above their 200 DMAs The Economic Times <|endoftext|> Maruti Suzuki
--------------------------------------------------------------------------------


## Summary

You built an end-to-end applied LLM system:

- **Collected** live Nifty 50 news from the internet (Google News RSS +
  Moneycontrol/LiveMint feeds) using only the standard library.
- **Trained a GPT from scratch** on that corpus — it learned to produce
  stock-news-style text.
- **Answered questions** about the stocks with a from-scratch TF-IDF **retrieval**
  layer (RAG), giving grounded, sourced answers, plus a retrieval+generation combo.

### Why RAG for the Q&A?
Small models can't memorize facts reliably, and news changes daily. Retrieval
keeps answers **accurate and current** without retraining — the same reason
real assistants use RAG.

### Scale it up
- On a GPU, all 50 stocks are fetched and a larger model trains longer (set above
  automatically). For even better generation, gather more history, increase
  `emb_dim`/`n_layers`, and train more `MAX_STEPS`.
- Swap the from-scratch model for a pretrained GPT-2 (as in Chapter 5) and
  instruction-tune it (Chapter 7) on `(question, answer-from-news)` pairs for a
  true generative financial assistant.

**This is not investment advice — it's an educational NLP project.**